In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [56]:
# Custom Dataset class
class DeepLenseDataset(Dataset):
    def __init__(self, data_dir, split='train', categories=['no', 'sphere', 'vort']):
        self.data_dir = data_dir
        self.categories = categories
        self.images = []
        self.labels = []
        self.class_to_idx = {cat: idx for idx, cat in enumerate(categories)}
        
        # Load data
        for cat_idx, category in enumerate(categories):
            cat_dir = os.path.join(data_dir, split, category)
            files = sorted([f for f in os.listdir(cat_dir) if f.endswith('.npy')])
            for file in files:
                self.images.append(os.path.join(cat_dir, file))
                self.labels.append(cat_idx)
        
        print(f"Loaded {len(self.images)} images from {split} set")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        img = np.load(img_path).astype(np.float32)
        # Remove channel dimension if needed and add it back with 1 channel
        if img.ndim == 3:
            img = img[0]  # Take first channel
        img = np.expand_dims(img, axis=0)  # Add channel dimension
        
        # Normalize: standardize to mean=0, std=1
        img = (img - img.mean()) / (img.std() + 1e-8)
        
        return torch.from_numpy(img), label

# Create datasets
train_dataset = DeepLenseDataset('dataset', split='train')
val_dataset = DeepLenseDataset('dataset', split='val')

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Loaded 30000 images from train set
Loaded 7500 images from val set
Train batches: 938
Val batches: 235


In [57]:
# CNN Model
class DeepLenseCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(DeepLenseCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        # Calculate flattened size: after 4 max pools (2^4 = 16), 150/16 ≈ 9, so 9x9x256
        self.fc1 = nn.Linear(256 * 9 * 9, 512)
        # Calculate flattened size: after 2 max pools (2^2 = 4), 150/4 ≈ 37, so 18x18x256
        # self.fc1 = nn.Linear(256 * 37 * 37, 1024)
        self.fc2 = nn.Linear(512, 256)

        # self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        # Conv block 1
        # print ("Shape before conv block 1 -", x.shape)

        # x = torch.relu(self.bn1(self.conv1(x)))
        x = F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.01)

        x = self.pool(x)
        # print ("Shape after conv block 1 -", x.shape)
        
        # Conv block 2
        # x = torch.relu(self.bn2(self.conv2(x)))
        x = F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.01)

        x = self.pool(x)
        # print ("Shape after conv block 2 -", x.shape)

        # Conv block 3
        # x = torch.relu(self.bn3(self.conv3(x)))
        x = F.leaky_relu(self.bn3(self.conv3(x)), negative_slope=0.01)

        x = self.pool(x)
        # print ("Shape after conv block 3 -", x.shape)
        
        # Conv block 4
        # x = torch.relu(self.bn4(self.conv4(x)))
        x = F.leaky_relu(self.bn4(self.conv4(x)), negative_slope=0.01)

        x = self.pool(x)
        # print ("Shape after conv block 4 -", x.shape)
        
        # print ("Shape after conv layers--", x.shape)
        # Flatten
        x = x.view(x.size(0), -1)
        # print ("Shape after flatten layer--", x.shape)
        # Fully connected layers
        # x = torch.relu(self.fc1(x))
        x = F.leaky_relu(self.fc1(x), negative_slope=0.01)

        x = self.dropout(x)
        # x = torch.relu(self.fc2(x))
        x = F.leaky_relu(self.fc2(x), negative_slope=0.01)

        x = self.dropout(x)
        x = self.fc3(x)
        
        return x


# Initialize model
model = DeepLenseCNN(num_classes=3).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

Model created with 11138243 parameters


In [58]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc="Training")
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        # print (outputs, predicted, labels)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({'Loss': loss.item(), 'Acc': 100 * correct / total})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc="Validating")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            progress_bar.set_postfix({'Loss': loss.item(), 'Acc': 100 * correct / total})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [59]:
# Training loop
num_epochs = 10
train_losses = []
train_accs = []
val_losses = []
val_accs = []
best_val_acc = 0
patience_counter = 0
early_stopping_patience = 3

print("Starting training...")
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    val_loss, val_acc, preds, labels = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        print(f"Best model saved! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1

    # Get predictions on validation set
    _, _, final_preds, final_labels = validate(model, val_loader, criterion, device)

    # Classification report
    categories = ['no', 'sphere', 'vort']
    print("\nClassification Report:")
    print(classification_report(final_labels, final_preds, target_names=categories))
    
    # Early stopping
    if patience_counter >= early_stopping_patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("\nTraining completed!")

# Load best model
model.load_state_dict(torch.load('best_model.pth'))
print("Best model loaded.")

Starting training...

Epoch 1/10


Starting training...

Epoch 1/10


Validating: 100%|██████████| 235/235 [00:08<00:00, 28.52it/s, Loss=1.07, Acc=33.3] 


Starting training...

Epoch 1/10


Validating: 100%|██████████| 235/235 [00:08<00:00, 28.52it/s, Loss=1.07, Acc=33.3] 


...


Classification Report:
              precision    recall  f1-score   support

          no       0.33      1.00      0.50      2500
      sphere       0.00      0.00      0.00      2500
        vort       0.00      0.00      0.00      2500

    accuracy                           0.33      7500
   macro avg       0.11      0.33      0.17      7500
weighted avg       0.11      0.33      0.17      7500


Epoch 4/10


Validating: 100%|██████████| 235/235 [00:08<00:00, 26.34it/s, Loss=1.14, Acc=33.3]


Train Loss: 1.1025, Train Acc: 32.95%
Val Loss: 1.1002, Val Acc: 33.33%


Validating: 100%|██████████| 235/235 [00:08<00:00, 28.44it/s, Loss=1.14, Acc=33.3]
/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wit


Classification Report:
              precision    recall  f1-score   support

          no       0.00      0.00      0.00      2500
      sphere       0.33      1.00      0.50      2500
        vort       0.00      0.00      0.00      2500

    accuracy                           0.33      7500
   macro avg       0.11      0.33      0.17      7500
weighted avg       0.11      0.33      0.17      7500

Early stopping at epoch 4

Training completed!
Best model loaded.


In [53]:
# Get final predictions on validation set
_, _, final_preds, final_labels = validate(model, val_loader, criterion, device)

# Classification report
categories = ['no', 'sphere', 'vort']
print("\nClassification Report:")
print(classification_report(final_labels, final_preds, target_names=categories))

# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)
print("\nConfusion Matrix:")
print(cm)

Validating: 100%|██████████| 235/235 [00:08<00:00, 28.26it/s, Loss=1.02, Acc=33.3] 

Validating: 100%|██████████| 235/235 [00:08<00:00, 28.26it/s, Loss=1.02, Acc=33.3] 


Classification Report:
              precision    recall  f1-score   support

          no       0.00      0.00      0.00      2500
      sphere       0.00      0.00      0.00      2500
        vort       0.33      1.00      0.50      2500

    accuracy                           0.33      7500
   macro avg       0.11      0.33      0.17      7500
weighted avg       0.11      0.33      0.17      7500


Confusion Matrix:
[[   0    0 2500]
 [   0    0 2500]
 [   0    0 2500]]



/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/betterbrambola/miniconda3/envs/deeplense/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

In [ ]:
# Debug: Check class distribution and predictions
from collections import Counter
true_dist = Counter(final_labels)
pred_dist = Counter(final_preds)

print("\n=== Class Distribution ===")
print(f"True distribution: {dict(true_dist)}")
print(f"Predicted distribution: {dict(pred_dist)}")
print(f"\nCategories mapping: 0=no, 1=sphere, 2=vortex")

# Check if model is confident about predictions
model.eval()
all_confidences = []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        max_probs = torch.max(probs, dim=1)[0]
        all_confidences.extend(max_probs.cpu().numpy())

avg_confidence = np.mean(all_confidences)
print(f"\nAverage prediction confidence: {avg_confidence:.4f}")
print(f"Min confidence: {np.min(all_confidences):.4f}, Max: {np.max(all_confidences):.4f}")

In [54]:
# Compute ROC curve and AUC score
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

# Get probability predictions
model.eval()
all_probs = []
all_labels_roc = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_labels_roc.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_labels_roc = np.array(all_labels_roc)

# Binarize labels for multiclass ROC
num_classes = 3
all_labels_binarized = label_binarize(all_labels_roc, classes=[0, 1, 2])

# Compute ROC curve and AUC for each class (one-vs-rest)
fpr = {}
tpr = {}
roc_auc_per_class = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(all_labels_binarized[:, i], all_probs[:, i])
    roc_auc_per_class[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and AUC
fpr_micro, tpr_micro, _ = roc_curve(all_labels_binarized.ravel(), all_probs.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)

# Compute macro-average ROC AUC
roc_auc_macro = np.mean(list(roc_auc_per_class.values()))

print("\n=== ROC Curve and AUC Scores ===")
print(f"AUC (Micro-average): {roc_auc_micro:.4f}")
print(f"AUC (Macro-average): {roc_auc_macro:.4f}")
for i, cat in enumerate(categories):
    print(f"AUC ({cat}): {roc_auc_per_class[i]:.4f}")


=== ROC Curve and AUC Scores ===
AUC (Micro-average): 0.4990
AUC (Macro-average): 0.4970
AUC (no): 0.4911
AUC (sphere): 0.5021
AUC (vort): 0.4977


In [ ]:
# Plot ROC curves
colors = ['blue', 'red', 'green']
fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC curve for each class
for i, cat in enumerate(categories):
    ax.plot(fpr[i], tpr[i], color=colors[i], lw=2, 
            label=f'ROC {cat} (AUC = {roc_auc_per_class[i]:.4f})')

# Plot micro-average ROC curve
ax.plot(fpr_micro, tpr_micro, color='deepskyblue', linestyle='--', lw=2,
        label=f'Micro-average (AUC = {roc_auc_micro:.4f})')

# Plot diagonal line (random classifier)
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Multiclass Classification')
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

print("ROC curves saved as 'roc_curves.png'")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(train_accs, label='Train Accuracy')
axes[1].plot(val_accs, label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

print("Training curves saved as 'training_curves.png'")

In [ ]:
# Plot confusion matrix
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix - Validation Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

print("Confusion matrix saved as 'confusion_matrix.png'")